In [114]:
print('Hello Nemmo!!')

Hello Nemmo!!


In [115]:
import pandas as pd
import numpy as np
import json
import janitor
import warnings 
warnings.filterwarnings('ignore')
from eda_helpers import plot_correlation_matrix, fill_missing_by_correlation

In [116]:
df = pd.read_csv('../data/raw/train.csv')
df.head()

,id,age,daily_screen_time_hours,social_media_hours,gaming_hours,work_study_hours,sleep_hours,notifications_per_day,app_opens_per_day,weekend_screen_time,gender,stress_level,academic_work_impact,addicted_label
0,0,24.0,NaN,1.83,1.59,2.11,7.46,122.0,38.0,8.63,Male,Medium,No,1
1,1,19.0,5.97,1.08,NaN,3.03,8.22,76.0,19.0,NaN,Female,Medium,No,0
2,2,18.0,5.09,NaN,NaN,NaN,6.25,134.0,60.0,7.47,Female,Low,Yes,0
3,3,21.0,6.42,1.26,1.42,3.36,8.85,112.0,94.0,8.66,Other,Low,NaN,1
4,4,26.0,11.20,1.87,2.81,1.95,5.25,NaN,NaN,13.39,Female,Medium,No,1


In [117]:
df["gender"] = df["gender"].fillna("Unknown")

In [ ]:
# Confirm what was dropped, before running get_dummies
original_categories = sorted(df["gender"].dropna().unique())
dropped_category = original_categories[0]
print(f"Dropped category: {dropped_category}")

Dropped category: Female


In [ ]:
encoding_metadata = {
    "gender": {
        "method": "one_hot",
        "categories": original_categories,
        "dropped_baseline": dropped_category,
        "resulting_columns": [c for c in df.columns if c.startswith("gender_")],
    }
}

with open("../models/encoding_metadata.json", "w") as f:
    json.dump(encoding_metadata, f, indent=2)

In [ ]:
df = pd.get_dummies(df, columns=["gender"], prefix="gender", drop_first=True)

# now the gender_* columns actually exist — update and re-save metadata
encoding_metadata["gender"]["resulting_columns"] = [c for c in df.columns if c.startswith("gender_")]

with open("../models/encoding_metadata.json", "w") as f:
    json.dump(encoding_metadata, f, indent=2)

gender_cols = [c for c in df.columns if c.startswith("gender_")]
df[gender_cols] = df[gender_cols].astype(int)

print(df.filter(like="gender_").head())

   gender_Male  gender_Other  gender_Unknown
0            1             0               0
1            0             0               0
2            0             0               0
3            0             1               0
4            0             0               0


In [121]:
df['academic_work_impact'] = df['academic_work_impact'].map({"Yes": 1, "No": 0})

In [122]:
df["stress_level"] = df["stress_level"].map({"Low": 0, "Medium": 1, "High": 2})

In [ ]:
for col in df.columns:
    missing_count = df[col].isnull().sum()
    if missing_count > 0:
        perc_missing = (missing_count / len(df)) * 100
        print(f"Column '{col}' has {missing_count} missing values ({perc_missing:.2f}%).")
        df, msg = fill_missing_by_correlation(df, col, threshold=0.0)
        print(msg)
        perc_missing_after = (df[col].isnull().sum() / len(df)) * 100
        print(f"After filling, column '{col}' has {df[col].isnull().sum()} missing values ({perc_missing_after:.2f}%).")
        print("\n")


Column 'age' has 28929 missing values (4.18%).
Filled 19672 missing values in 'age' using ['weekend_screen_time', 'daily_screen_time_hours'] (LinearRegression). 9257 rows still missing (predictors were also null).
After filling, column 'age' has 9257 missing values (1.34%).


Column 'daily_screen_time_hours' has 95854 missing values (13.86%).
Filled 61400 missing values in 'daily_screen_time_hours' using ['weekend_screen_time', 'addicted_label'] (LinearRegression). 34454 rows still missing (predictors were also null).
After filling, column 'daily_screen_time_hours' has 34454 missing values (4.98%).


Column 'social_media_hours' has 133995 missing values (19.38%).
Filled 118772 missing values in 'social_media_hours' using ['daily_screen_time_hours', 'addicted_label'] (LinearRegression). 15223 rows still missing (predictors were also null).
After filling, column 'social_media_hours' has 15223 missing values (2.20%).


Column 'gaming_hours' has 126821 missing values (18.34%).
Filled 81909

In [125]:
# Print the percentage of missing rows in the dataframe 
perc_missing_rows = (df.isnull().any(axis=1).sum() / len(df)) * 100
print(f"Percentage of rows with missing values: {perc_missing_rows:.2f}%")

Percentage of rows with missing values: 10.69%


In [126]:
# Fill missing values with meadian of the feature 
for col in df.columns:
    if df[col].isnull().sum() > 0:
        median_value = df[col].median()
        df[col].fillna(median_value, inplace=True)
        print(f"Filled missing values in column '{col}' with median value {median_value}.")

Filled missing values in column 'age' with median value 26.78329323474305.
Filled missing values in column 'daily_screen_time_hours' with median value 7.794936929913353.
Filled missing values in column 'social_media_hours' with median value 2.41.
Filled missing values in column 'gaming_hours' with median value 1.36.
Filled missing values in column 'work_study_hours' with median value 2.22.
Filled missing values in column 'sleep_hours' with median value 6.834601034374806.
Filled missing values in column 'notifications_per_day' with median value 146.84384175175026.
Filled missing values in column 'app_opens_per_day' with median value 104.08864013428143.
Filled missing values in column 'weekend_screen_time' with median value 9.59.
Filled missing values in column 'stress_level' with median value 1.0.
Filled missing values in column 'academic_work_impact' with median value 0.5170832953914775.


In [127]:
df.to_csv("../data/processed/cleaned_data.csv", index=False)